# Sesionización de Clickstream con Apache Hadoop MapReduce (End-to-End)

**Materia:** Big Data (Maestría en IA)  
**Objetivo del notebook:** explicar de forma integral, paso a paso, cómo se implementó la sesionización de clickstream con Hadoop Streaming y Python.


## 1. Problema a resolver

Se requiere procesar eventos de navegación (clickstream) para:
- ordenar eventos por **usuario** y **tiempo**
- detectar **cortes de sesión** por inactividad
- calcular métricas por sesión (duración, páginas, ruta)
- detectar patrones atípicos
- construir una **tabla agregada por usuario/sesión** para analítica posterior

La solución implementada usa **Apache Hadoop + MapReduce (Streaming)** con scripts en Python.


## 2. Conceptos clave de Apache Hadoop

### 2.1 HDFS (almacenamiento distribuido)
- **NameNode**: mantiene metadatos del sistema de archivos distribuido (HDFS).
- **DataNode**: almacena físicamente los bloques de datos.

### 2.2 YARN (gestión de recursos y ejecución)
- **ResourceManager**: coordina recursos de cómputo del clúster.
- **NodeManager**: ejecuta contenedores/tareas en nodos de trabajo.
- **HistoryServer**: historial de jobs.

### 2.3 MapReduce
Modelo de procesamiento en dos fases:
1. **Map**: transforma registros de entrada en pares clave-valor.
2. **Reduce**: agrega/procesa registros agrupados por clave.

Entre ambas fases, Hadoop hace automáticamente **shuffle + sort** (agrupa y ordena por clave).

### 2.4 Hadoop Streaming
Permite usar ejecutables externos (por ejemplo, scripts Python) como mapper/reducer.


## 3. Arquitectura usada en esta práctica

Se levantó un clúster Hadoop con Docker Compose (`docker-compose.yml`) con estos servicios:
- `hadoop-namenode`
- `hadoop-datanode`
- `hadoop-resourcemanager`
- `hadoop-nodemanager`
- `hadoop-historyserver`
- `hadoop-client`

Además, se montó el proyecto dentro de contenedores en `/workspace` para ejecutar scripts directamente sobre el clúster.


## 4. Estructura de la solución implementada

Archivos principales:
- `mapreduce/mapper_clickstream.py`
- `mapreduce/reducer_sessionize.py`
- `mapreduce/mapper_user_agg.py`
- `mapreduce/reducer_user_agg.py`
- `run_hadoop_streaming.sh`
- `run_local_smoke.sh`
- `data/clickstream_sample.csv`
- `README.md`


## 5. Datos de entrada

El input se trabaja como CSV con cabeceras.

Columnas mínimas requeridas:
- `user_id`
- `event_time`
- `page`

Columnas opcionales:
- `event_type`
- `device`

Ejemplo de fila:

`1,u1,2026-02-20T10:00:00Z,/home,view,mobile`


## 6. Job 1: Sesionización de clickstream

### 6.1 Mapper (`mapper_clickstream.py`)
Responsabilidades:
- parsear el CSV
- normalizar `event_time` a epoch
- limpiar texto
- emitir salida tabular: `user_id, epoch, event_iso, page, event_type, device`

Clave lógica: `user_id + epoch`, para habilitar orden temporal por usuario en el reduce.

### 6.2 Reducer (`reducer_sessionize.py`)
Responsabilidades:
- leer eventos ordenados por usuario y tiempo
- cortar sesión cuando el gap entre eventos supera `SESSION_GAP_MINUTES` (default 30 min)
- calcular métricas por sesión

Métricas producidas por sesión:
- `session_id`
- `session_start`, `session_end`
- `duration_seconds`
- `pageviews`, `unique_pages`
- `entry_page`, `exit_page`
- `path_sequence`
- `anomaly_flags`

Reglas de anomalía implementadas:
- `bounce`: 1 página y duración <= 10 s
- `single_page`: 1 página y duración > 10 s
- `long_session`: duración >= 7200 s
- `rapid_clicking`: gap mínimo <= 2 s y al menos 5 eventos
- `high_depth`: >= 25 páginas
- `normal`: sin anomalías


## 7. Job 2: Tabla agregada por usuario

### 7.1 Mapper (`mapper_user_agg.py`)
Convierte cada sesión en una fila agregable por usuario:
- contador de sesiones
- duración
- pageviews
- bandera de sesión anómala

### 7.2 Reducer (`reducer_user_agg.py`)
Agrega por `user_id` y calcula:
- `total_sessions`
- `total_duration_seconds`
- `total_pageviews`
- `avg_session_duration_seconds`
- `avg_pages_per_session`
- `anomalous_sessions`
- `anomalous_rate`


## 8. Ejecución end-to-end

Comando ejecutado (dentro de `hadoop-namenode`):


```
docker exec -it hadoop-namenode bash -lc "cd /workspace && ./run_hadoop_streaming.sh"
```

Qué hace `run_hadoop_streaming.sh`:
1. detecta binarios (`hadoop`, `hdfs`, `python`, `streaming.jar`)
2. sube input a HDFS (`/input/clickstream/clickstream.csv`)
3. limpia outputs previos
4. ejecuta Job 1 (sesionización)
5. ejecuta Job 2 (agregado por usuario)
6. imprime muestra de resultados


## 9. Resultados logrados

### 9.1 Tabla de sesiones (HDFS)
Ruta: `hdfs:///output/clickstream_sessions`

Ejemplo de salida:
- `u1 ... duration=480 pageviews=3 anomaly=normal`
- `u3 ... duration=5 pageviews=6 anomaly=rapid_clicking`
- `u4 ... duration=7500 pageviews=6 anomaly=long_session`

### 9.2 Tabla agregada por usuario (HDFS)
Ruta: `hdfs:///output/clickstream_user_metrics`

Ejemplo de salida:
- `u1	2	480	4	240.00	2.00	1	0.5000`
- `u4	1	7500	6	7500.00	6.00	1	1.0000`

Con esto se obtiene una base lista para BI/analítica, scoring, segmentación o modelado posterior.


## 10. Troubleshooting real del proceso (aprendizajes)

Durante la implementación aparecieron problemas típicos de entornos Hadoop en contenedor:

1. **Contenedor client no persistente**
- Se ajustó el `command` para mantenerlo vivo y facilitar `docker exec`.

2. **Selección incorrecta del JAR de streaming**
- Se evitó usar `test-sources.jar` y se seleccionó correctamente:
  `.../share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar`

3. **Compatibilidad de opciones streaming**
- En esta imagen se usó `-file` (aunque aparezca deprecado) para empaquetar scripts de mapper/reducer.

4. **Ausencia de Python en imagen base**
- Se instaló Python dentro de `namenode` y el script autodetecta el interprete disponible.

5. **Compatibilidad de scripts**
- Scripts adaptados a sintaxis compatible con Python 2/3 para mayor portabilidad.


## 11. ¿Qué se logró exactamente?

Se entregó una implementación funcional end-to-end de sesionización con Hadoop MapReduce que:
- procesa clickstream en HDFS
- aplica reglas de corte de sesión por inactividad
- calcula métricas de comportamiento por sesión
- detecta patrones atípicos
- genera tabla agregada por usuario
- deja resultados listos para analítica posterior

Además, se documentó y automatizó la ejecución para que pueda repetirse con datasets reales cambiando solo el archivo de entrada y parámetros.


## 12. Próximos pasos recomendados

- Persistir Python en una imagen personalizada (Dockerfile) para no reinstalar en cada reinicio.
- Forzar ejecución distribuida YARN (si la rúbrica exige evitar modo local).
- Añadir más reglas de anomalía (por dispositivo, hora, rutas raras, etc.).
- Guardar resultados en formato columna (Parquet/ORC) para consumo analítico más eficiente.
